# Accuracy Comparison

In [1]:
# Loading libraries

import numpy as np
import pandas as pd

In [2]:
# Importing data

df = pd.read_csv('predictions_data.csv')
df.drop('Unnamed: 0', axis=1, inplace=True)
df.info()
df.sample(10, random_state = 123)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Variant   72 non-null     object 
 1   Index     72 non-null     object 
 2   Approach  72 non-null     object 
 3   Model     72 non-null     object 
 4   Leader    72 non-null     object 
 5   Forecast  72 non-null     float64
dtypes: float64(1), object(5)
memory usage: 3.5+ KB


,Variant,Index,Approach,Model,Leader,Forecast
63,penalized,YHRM,BERT,LSTM,fayulu,0.351835
26,baseline,YHRM,BERT,ARIMA,tshisekedi,0.357116
71,penalized,YHRM,VADER,LSTM,tshisekedi,0.348654
65,penalized,YHRM,BERT,LSTM,tshisekedi,0.348907
23,baseline,WG,VADER,LSTM,tshisekedi,0.128092
8,baseline,TSSW,VADER,ARIMA,tshisekedi,0.318231
52,penalized,WG,BERT,LSTM,ramazani,0.261510
64,penalized,YHRM,BERT,LSTM,ramazani,0.267339
29,baseline,YHRM,BERT,LSTM,tshisekedi,0.354086
54,penalized,WG,VADER,ARIMA,fayulu,0.599391


In [3]:
# Observed vote shares from CENI data (benchmark)
observed = {'fayulu': 0.3482, 'ramazani': 0.2383, 'tshisekedi': 0.3856}

In [4]:
# Accuracy function

def accuracy_score(observed_dict, predicted_dict):
    """
    Calculate accuracy score using the relative-absolute formula.
    
    Parameters:
    observed_dict (dict): Observed vote shares {candidate: share}
    predicted_dict (dict): Predicted vote shares {candidate: share}
    
    Returns:
    float: Accuracy score between 0 and 1
    """
    K = len(observed_dict)  # Number of candidates
    total_error = 0
    
    for candidate, observed_share in observed_dict.items():
        if candidate in predicted_dict:
            predicted_share = predicted_dict[candidate]
            # Avoid division by zero
            if observed_share > 0:
                relative_error = abs(predicted_share - observed_share) / observed_share
                total_error += relative_error
    
    accuracy = 1 - (total_error / K)
    return accuracy

In [5]:
# Create accuracy results
accuracy_results = []

# Iterate through all combinations in final_df
for (variant, index, approach, model), group in df.groupby(['Variant', 'Index', 'Approach', 'Model']):
    # Convert group to predicted dictionary
    predicted_dict = dict(zip(group['Leader'], group['Forecast']))
    
    # Calculate accuracy
    accuracy_values = accuracy_score(observed, predicted_dict)
    
    accuracy_results.append({
        'Variant': variant,
        'Index': index,
        'Approach': approach,
        'Model': model,
        'Accuracy': accuracy_values
    })

# Create accuracy dataframe
accuracy_df = pd.DataFrame(accuracy_results)

In [6]:
# Pivot to get the desired format: indices as columns, variant-approach-model combinations as rows
accuracy_pivot = accuracy_df.pivot_table(
    index=['Model', 'Approach', 'Variant'],
    columns='Index',
    values='Accuracy'
).reset_index()

# Sort for consistent ordering
accuracy_pivot = accuracy_pivot.sort_values(['Model', 'Approach', 'Variant'])

# Rename the index to combine variant, approach, and model
accuracy_pivot['Method'] = accuracy_pivot['Variant'] + '_' + accuracy_pivot['Approach'] + '_' + accuracy_pivot['Model']
accuracy_pivot = accuracy_pivot[['Method', 'TSSW', 'WG', 'YHRM']]

print(f"{'='*50}\nFinal Accuracy Table:\n{'='*50}")
accuracy_pivot.round(4)

Final Accuracy Table:


Index,Method,TSSW,WG,YHRM
0,baseline_BERT_ARIMA,0.8251,0.5634,0.9067
1,penalized_BERT_ARIMA,0.8096,0.6401,0.9140
2,baseline_VADER_ARIMA,0.8178,0.5425,0.9362
3,penalized_VADER_ARIMA,0.9487,0.5379,0.9314
4,baseline_BERT_LSTM,0.8679,0.5104,0.8361
5,penalized_BERT_LSTM,0.9090,0.5008,0.9242
6,baseline_VADER_LSTM,0.7381,0.4674,0.7706
7,penalized_VADER_LSTM,0.7469,0.4691,0.9119


In [7]:
# Export to CSV
csv_filename = "accuracy_data.csv"
accuracy_pivot.to_csv(csv_filename)
print(f"\nDataFrame successfully exported to: {csv_filename}")


DataFrame successfully exported to: accuracy_data.csv
